# Where does a well's water come from?

A water utility pumping from a valley aquifer has neighbors: a lake at one end,
rain falling on the whole valley, and the water that was already underground
when pumping started. A regulator asking whether the wells are drawing lake
water, and a manager asking how long the aquifer's own storage will last, are
asking the same question in two ways. Heads cannot answer it, because every drop
of water in the aquifer has the same head whatever its origin.

## What this notebook covers

Label each source of water with a tracer and let the transport model carry the
labels to the wells. The concentration that arrives is the fraction of the well's
water that came from that source.

By the end of this notebook you will be able to:

- set up an **end-member** simulation, where one source is given a concentration
  of 1 and everything else 0,
- tell a source apart from an initial condition, which needs a different kind of
  run,
- read source fractions from observations at the pumping wells, and
- check the result by adding the fractions up.

The model is the Voronoi-grid synthetic valley that ships with this repository,
the same one [`flopy-intro-gwt-a`](flopy-intro-gwt-a.ipynb) and
[`flopy-intro-gwt-b`](flopy-intro-gwt-b.ipynb) use, and it is the MODFLOW 6
example [ex-gwt-synthetic-valley](https://modflow6-examples.readthedocs.io/en/master/_examples/ex-gwt-synthetic-valley.html).
It is in meters and days.

Import the packages this notebook uses. `mf6_gwt_source_water` holds the model
builders, so the cells below stay about the water rather than about flopy.

In [ ]:
%matplotlib inline

import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import mf6_gwt_source_water as sw
import numpy as np
from mf6_notebook_helpers import find_mf6_libraries

_, mf6_exe = find_mf6_libraries()

## The idea: label the water

A tracer that does not decay, sorb, or change the flow is a label. Put a
concentration of 1 on one source and 0 on the others, and every water molecule
carries a number saying whether it came from that source. Mix the water in an
aquifer and the concentration at any point becomes the fraction of the water at
that point that started at the labeled source.

That gives one simulation per source. Here there are three sources, so three
runs:

| end member | how it is labeled |
|---|---|
| lake | the lake package's concentration auxiliary is 1 |
| recharge | the recharge package's concentration auxiliary is 1 |
| initial | the aquifer starts at a concentration of 1 everywhere |

The third is different in kind. Water already in the aquifer is not a source
pouring in at a boundary; it is what the model starts with, so it is labeled
through the initial condition instead.

Each end member needs its own flow run as well as its own transport run, because
a source concentration is an auxiliary variable on a flow package and it reaches
transport through the flow model's budget file.

In [ ]:
for name, (lake, recharge, initial) in sw.END_MEMBERS.items():
    print(
        f"{name:9s} lake {lake:.0f}   recharge {recharge:.0f}   initial {initial:.0f}"
    )
print()
print(f"pumping period: {sw.PERLEN / 365.25:.0f} years in {sw.NSTP} steps")
print(f"wells:          {', '.join(sw.WELL_NAMES)}")

## Run the three end members

Each run is a flow model followed by a transport model that reads it. The flow
model starts from a steady period with the wells off, then pumps for 30 years;
the transport model follows the same steps, because the Flow Model Interface
reads one flow budget record per transport time step.

Six simulations, about two minutes.

In [ ]:
model_root = pl.Path("models/gwt-source-water")
fractions = sw.source_fractions(model_root, mf6_exe)

years = fractions["lake"]["time"] / 365.25
print(f"output times: {len(years)}, ending at {years.iloc[-1]:.0f} years")

## What the wells produce

The concentration at each well is that end member's share of the water it
pumps. Read the three runs at the end of the simulation.

In [ ]:
final = sw.final_fractions(fractions)
final["total"] = final.sum(axis=1)
print(final.to_string(float_format=lambda v: f"{v:8.4f}"))

The totals are the check worth making. The three sources are the only water the
wells can draw, so the fractions have to add to 1 at every well and at every
time. Anything else means a source is missing, or a concentration was set on the
wrong package.

In [ ]:
print(f"largest departure from 1: {sw.fraction_closure(fractions):.2e}")

Plot the three fractions through time at each well, stacked so the total shows.

In [ ]:
colors = {"lake": "tab:blue", "recharge": "tab:green", "initial": "tab:brown"}

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [list("ABC")], figsize=(9.5, 3.6), layout="constrained", sharey=True
    )
    for letter, well in zip("ABC", sw.WELL_NAMES):
        ax = axd[letter]
        ax.stackplot(
            years,
            [fractions[member][well] for member in sw.END_MEMBERS],
            labels=list(sw.END_MEMBERS),
            colors=[colors[member] for member in sw.END_MEMBERS],
        )
        ax.set_xlim(0.0, years.iloc[-1])
        ax.set_ylim(0.0, 1.0)
        ax.set_xlabel("time, in years")
        ax.set_title(f"{letter}. Well {well}")
    axd["A"].set_ylabel("fraction of the water pumped")
    axd["C"].legend(loc="lower right", fontsize=7)

**What to look for.** Each band is one source, and the three fill the panel
because they are the only water there is. Green dominates every well: after 30
years, recharge supplies 0.93 of what P1 produces, 0.82 of P2, and 0.92 of P3.
The brown band is the water that was in the aquifer when pumping started, and it
shrinks from about half at 10 years to between 0.07 and 0.18 at 30.

There is no blue band. Lake water never reaches these wells: the largest lake
fraction at any well at any time is 2e-5, which is nothing. A well 4 km from a
lake, in an aquifer where water moves at a fraction of a meter a day, is not
pumping lake water 30 years later, whatever a map of the valley might suggest.

## Where the labeled water goes

Reading the wells says how much lake water arrives but not why so little does.
Map two end members in the layer the wells are screened in: the lake water, and
the recharge that supplies nearly all of what the wells produce.

In [ ]:
flow_sim = flopy.mf6.MFSimulation.load(
    sim_ws=str(model_root / "lake-flow"),
    verbosity_level=0,
    load_only=["disv", "well-1"],
)
gwf = flow_sim.get_model()
wells = gwf.get_package("well-1").stress_period_data.get_data(1)
well_cells = {
    str(name).upper(): tuple(cellid)
    for name, cellid in zip(wells["boundname"], wells["cellid"])
}
well_layer = next(iter(well_cells.values()))[0]


def end_member_map(end_member):
    """Concentration in the well layer at the end of the run."""
    path = model_root / f"{end_member}-transport" / f"{sw.GWT_NAME}.ucn"
    conc = flopy.utils.HeadFile(path, text="CONCENTRATION").get_data()
    # one value per cell in the layer, with MODFLOW's placeholder for the
    # inactive cells taken out
    layer = conc[well_layer].ravel()
    return np.where(layer > 1.0e29, np.nan, layer)


with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"]], figsize=(9.0, 6.5), layout="constrained"
    )
    for letter, end_member in zip("AB", ("lake", "recharge")):
        ax = axd[letter]
        values = end_member_map(end_member)
        pmv = flopy.plot.PlotMapView(gwf, ax=ax, layer=well_layer)
        patches = pmv.plot_array(values, cmap="viridis", vmin=0.0, vmax=1.0)
        pmv.plot_grid(linewidth=0.15, color="0.9")
        for well, cellid in well_cells.items():
            x = gwf.modelgrid.xcellcenters[cellid[1]]
            y = gwf.modelgrid.ycellcenters[cellid[1]]
            ax.plot(x, y, "o", color="red", ms=5)
            ax.annotate(
                well, (x, y), xytext=(5, 5), textcoords="offset points", fontsize=8
            )
        ax.set_xlabel("x, in meters")
        ax.set_ylabel("y, in meters")
        ax.set_title(f"{letter}. {end_member.capitalize()} water")
    fig.colorbar(
        patches,
        ax=list(axd.values()),
        shrink=0.5,
        label=f"fraction of the water in layer {well_layer + 1}",
    )

**What to look for.** Panel A is almost empty. Lake water has spread into the
aquifer beneath and around the lake, reaching a fraction of 0.10 in this layer
at its strongest, but it stops well short of the wells at the southern end of
the valley. Panel B is the opposite: recharge has replaced nearly all the water
in the pumped layer, reaching 0.94 at its strongest.

The two panels together explain the stacked bars. The wells sit in the part of
the aquifer that recharge fills, and 30 years is not long enough for water to
travel from the lake to them.

## Recap

- Labeling a source with a tracer concentration of 1 turns a transport model
  into a way of measuring where a well's water comes from, which heads alone
  cannot say.
- Each source needs its own simulation, and water already in the aquifer is
  labeled through the initial condition rather than through a boundary.
- The fractions at a well have to add to 1, and checking that is how you know
  the set of sources is complete. Here they close to within 4e-6.
- After 30 years of pumping, these wells produce 0.82 to 0.93 recharge, the rest
  water that was already in the aquifer, and no measurable lake water at all.